# **Environment Setup**

In [ ]:
%pip install -q -U groq

In [ ]:
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    from getpass import getpass
    GROQ_API_KEY = getpass('Paste your Groq API key (input hidden): ')

client = Groq(api_key=GROQ_API_KEY)

# Query the models actually available to this key rather than hardcoding one
available = client.models.list()
available_ids = [m.id for m in available.data]
print("Models available to your key:")
for mid in available_ids:
    print(" -", mid)

# Prefer models 
PREFERRED_MODELS = [
    "qwen/qwen3.8-27b",
    "qwen/qwen3.6-27b",
    "qwen/qwen3-32b",
    "moonshotai/kimi-k2-instruct-0905",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
]

def pick_model(exclude=None):
    """Picks the best available model from PREFERRED_MODELS, skipping anything in
    `exclude` (used to skip a model that just hit its daily quota). Falls back to any
    other available model if none of the preferred ones are left. Returns None if
    nothing is left to try. Reused at runtime by switch_to_next_model below, so a
    daily-quota error mid-session can fail over to a different model automatically."""
    exclude = exclude or set()
    candidates = [m for m in PREFERRED_MODELS if m in available_ids and m not in exclude]
    if not candidates:
        candidates = [m for m in available_ids if m not in exclude]
    return candidates[0] if candidates else None


def reasoning_effort_for(model_id):
    """Qwen models are hybrid 'thinking' models; reasoning_effort='none' suppresses
    visible <think> tags. gpt-oss models use a different low/medium/high scale.
    Other models get no reasoning_effort parameter at all."""
    if model_id and model_id.startswith("qwen/"):
        return "none"
    if model_id and "gpt-oss" in model_id:
        return "low"
    return None


MODEL_ID = pick_model()
REASONING_EFFORT = reasoning_effort_for(MODEL_ID)

print("\nSelected MODEL_ID:", MODEL_ID)
if MODEL_ID and "gpt-oss" in MODEL_ID:
    print("Note: gpt-oss models can raise a 'tool_use_failed' error once a system")
    print("prompt is introduced later in this notebook. Pick a different MODEL_ID above if so.")
print("If this model's daily quota runs out mid-session, the app will automatically")
print("switch to the next available model (see create_with_retry / switch_to_next_model).")

sanity_kwargs = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT else {}
response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is overfitting in machine learning?"}],
    max_tokens=100,
    **sanity_kwargs,
)
print(response.choices[0].message.content)

# **Baseline Chat (No Persona)**

In [ ]:
%pip install -q -U gradio

In [ ]:
import gradio as gr
print("Gradio version:", gr.__version__)

def basic_chat(message, history):
    """Plain baseline chat: no persona, no prompting technique. Normalizes Gradio's
    history format (dict or tuple, depending on version) into {'role', 'content'}
    dicts only, since Groq rejects unrecognized extra fields."""
    messages = []
    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn.get("role"), "content": turn.get("content")})
        else:
            user_msg, bot_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if bot_msg:
                messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=800,
    )
    return response.choices[0].message.content


demo = gr.ChatInterface(
    fn=basic_chat,
    title="The Interrogator — Baseline Chat",
    description="Plain baseline chat, no detective persona applied.",
)

# demo.launch(share=True, debug=True)  # launched in the Dynamic UI section 

# **Persona and Few-Shot Prompting**

In [ ]:
MAX_QUESTIONS = 5  # maximum follow-up questions before a case must close

FEW_SHOT_EXAMPLES = """
--- EXAMPLE 1: A vague opener with almost no evidence yet — ask, don't guess ---
User: My cat has been acting weird lately.
Detective:
CASE OPENED

EVIDENCE:
- Your cat's behavior has changed, but no specifics are given yet.

HYPOTHESES:
- H1: A physical health issue (pain, illness).
- H2: A stress or environmental change (new pet, moved furniture, schedule change).
- H3: A normal age-related behavior shift.

MISSING:
- What specifically changed (eating, hiding, aggression, vocalizing).
- How long this has been going on.

What exactly has changed about your cat's behavior, and when did you first notice it?

--- EXAMPLE 2: A technical report with real signal — form sharp, specific hypotheses ---
User: My web app returns intermittent 500 errors, only during peak traffic hours.
Detective:
CASE OPENED

EVIDENCE:
- 500 errors occur intermittently, correlated with peak traffic.

HYPOTHESES:
- H1: Resource exhaustion under load (DB connection pool, memory, CPU).
- H2: A race condition that only surfaces with concurrent requests.
- H3: A downstream dependency (API, cache, DB) timing out under load.

MISSING:
- Whether the errors cluster around one endpoint or are spread across the app.
- What the server logs show at the moment of failure.

When these 500s happen, do your server logs point to a specific endpoint or component, or does it look random across the app?

--- EXAMPLE 3: Enough evidence in one message — close immediately, no unnecessary questions ---
User: I dropped my phone in the toilet five minutes ago and now the screen is completely black and it won't turn on.
Detective:
CASE CLOSED

PRIMARY CONCLUSION: Water damage from the toilet drop has short-circuited or shut down your phone.

EVIDENCE: Direct liquid exposure five minutes ago, immediately followed by total screen failure — a textbook liquid-damage signature.

HYPOTHESES RULED OUT: A pre-existing battery or software fault is far less likely given the precise timing right after submersion.

RECOMMENDATION: Power it off if it isn't already, do not charge it, remove the case and SIM if possible, and leave it in a desiccant/silica-gel drying pouch for at least 24-48 hours before attempting to power on.
"""

SYSTEM_PROMPT = """You are THE INTERROGATOR, a sharp, methodical detective persona.

You do not answer questions directly. Instead, you treat every user message as a CASE.
Your job is to investigate the case, not to immediately solve it.

Behavior rules:
1. Every single response, without exception, must use these exact section
   headers in this order: for a new or ongoing case use CASE OPENED (first
   turn only) or CASE UPDATE (later turns), then EVIDENCE:, HYPOTHESES:,
   MISSING:, and end with your one question. Use these headers even for
   short, casual, or low-detail messages — do not drop the structure just
   because there isn't much evidence yet; in that case, say so explicitly
   under EVIDENCE: and MISSING:.
2. Identify the EVIDENCE already present in what the user told you.
3. Form one or more plausible HYPOTHESES based on that evidence.
4. Identify what key information is still MISSING.
5. Ask exactly ONE focused, high-value follow-up question to fill the most
   important gap. Never ask multiple questions in the same turn.
6. Continue this investigation across turns: update evidence and hypotheses
   as the user answers, and ask the next most useful question.
7. You will be told in a status note how many questions you have already
   asked. You may ask at most {max_q} follow-up questions total. Once that
   limit is reached (the status note will say so explicitly), you MUST NOT
   ask another question — you must close the case on that turn instead.
8. To close a case: say "CASE CLOSED," state your primary conclusion,
   briefly note the evidence that supports it, and mention any hypotheses
   you ruled out and why. Keep the closing summary tight — a few sentences
   per section, not an essay.
9. Stay in character. Use light detective language (case, evidence,
   hypothesis, lead, clue) naturally, but don't overdo it to the point of
   being hard to read.
10. If a case has already been closed and the user sends a new, clearly
    different problem to investigate, open a brand-new case from scratch
    (do not blend it with the old one). If the user is just acknowledging,
    thanking you, or confirming they'll follow your advice — for example
    "thanks", "okay", "got it", "okay i will do that", "sure" — that is
    NOT a new case, even though it sounds like agreement or future action.
    Reply briefly and naturally in character instead of opening a case.
    If you are genuinely unsure whether a message is a new case or just an
    acknowledgment, default to treating it as an acknowledgment.
11. Always address the person directly as "you." Never refer to them in
    the third person as "the user" — that breaks the conversational feel.
    Example: write "What were you doing right before it started?" not
    "What was the user doing before the headache started?"
12. The examples below show the expected style, structure, and judgment —
    including when to ask a question versus when there's already enough
    evidence to close immediately. Match this style and level of specificity,
    but never reuse or reference the example cases themselves; each new
    case from the user is independent.

EXAMPLES:
{few_shot}
""".format(max_q=MAX_QUESTIONS, few_shot=FEW_SHOT_EXAMPLES)


def extract_text(content):
    """Normalizes Gradio message content to plain text. Content can be a string
    or a list of parts (e.g. [{'type': 'text', 'text': '...'}]) depending on
    the Gradio version in use."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
        return "".join(parts)
    return "" if content is None else str(content)


def normalize_history(history):
    """Flattens Gradio history (dict or tuple format) into a list of (role, text) tuples."""
    normalized = []
    for turn in history:
        if isinstance(turn, dict):
            normalized.append((turn.get("role"), extract_text(turn.get("content"))))
        else:
            user_msg, bot_msg = turn
            if user_msg:
                normalized.append(("user", extract_text(user_msg)))
            if bot_msg:
                normalized.append(("assistant", extract_text(bot_msg)))
    return normalized


def count_questions_asked(history):
    """Counts assistant turns since the most recent case closure, so a new case
    does not inherit an already-exhausted question budget from a prior case."""
    normalized = normalize_history(history)

    last_close_idx = -1
    for i, (role, text) in enumerate(normalized):
        if role == "assistant" and "CASE CLOSED" in text.upper():
            last_close_idx = i

    count = 0
    for i, (role, text) in enumerate(normalized):
        if i > last_close_idx and role == "assistant":
            count += 1
    return count


def case_already_closed(history):
    """Returns whether the case is currently closed, based on the most recent
    assistant turn that carried a CASE OPENED/UPDATE/CLOSED header. Header-less
    acknowledgment replies (e.g. "glad I could help") do not change this state,
    which avoids mistaking small talk after a close for the case still being open."""
    state_closed = False
    for role, text in normalize_history(history):
        if role != "assistant":
            continue
        upper = text.upper().lstrip()
        if upper.startswith("CASE CLOSED"):
            state_closed = True
        elif upper.startswith("CASE OPENED") or upper.startswith("CASE UPDATE"):
            state_closed = False
    return state_closed


def history_since_last_close(history):
    """Returns the turns after the most recent CASE CLOSED, excluding any earlier,
    already-closed case from the same session."""
    normalized = normalize_history(history)
    last_close_idx = -1
    for i, (role, text) in enumerate(normalized):
        if role == "assistant" and text.upper().lstrip().startswith("CASE CLOSED"):
            last_close_idx = i
    return normalized[last_close_idx + 1:]


import re
import time

MAX_RETRY_WAIT = 25  # seconds; longer waits (such as a daily quota) are reported instead of retried


def rate_limit_wait(error):
    """Seconds to wait if the error is a short-lived rate limit (for example a per-minute
    cap), otherwise None. Long waits such as daily quotas are not retried."""
    message = str(error)
    if "rate_limit" not in message and "429" not in message:
        return None
    ms = re.search(r"try again in (\d+(?:\.\d+)?)ms", message)
    if ms:
        seconds = float(ms.group(1)) / 1000
    else:
        m = re.search(r"try again in (?:(\d+)m)?(?:(\d+(?:\.\d+)?)s)?", message)
        if not m or not (m.group(1) or m.group(2)):
            return None
        seconds = int(m.group(1) or 0) * 60 + float(m.group(2) or 0)
    return seconds + 0.5 if seconds <= MAX_RETRY_WAIT else None


def switch_to_next_model(exclude=None):
    """Fails over to the next available model when the current one hits a long-lived
    limit (e.g. a daily quota exhausted mid-session), instead of the app just breaking.
    Updates the global MODEL_ID and REASONING_EFFORT using the same preference order
    picked at startup (pick_model, from Environment Setup). Returns the new MODEL_ID,
    or None if every available model has already been tried."""
    global MODEL_ID, REASONING_EFFORT
    exclude = set(exclude or ())
    exclude.add(MODEL_ID)
    new_model = pick_model(exclude=exclude)
    if new_model is None:
        print("[model switch] no alternative model available \u2014 all options exhausted")
        return None
    print(f"[model switch] {MODEL_ID} hit a long-lived limit (likely a daily quota) \u2014 switching to {new_model}")
    MODEL_ID = new_model
    REASONING_EFFORT = reasoning_effort_for(new_model)
    return new_model


def create_with_retry(max_attempts=4, **kwargs):
    """chat.completions.create with automatic waits on short rate limits, and automatic
    fallback to a different model on long-lived limits (e.g. a daily quota running out
    mid-session) instead of the request just failing. model and reasoning_effort in
    kwargs are refreshed from the current globals on every attempt, so a mid-loop model
    switch is picked up immediately rather than retrying the exhausted model again."""
    tried_models = set()
    for attempt in range(max_attempts):
        kwargs["model"] = MODEL_ID
        kwargs.pop("reasoning_effort", None)
        if REASONING_EFFORT:
            kwargs["reasoning_effort"] = REASONING_EFFORT
        try:
            return client.chat.completions.create(**kwargs)
        except Exception as e:
            wait = rate_limit_wait(e)
            if wait is not None:
                print(f"[rate limit] waiting {wait:.0f}s before retrying")
                time.sleep(wait)
                continue
            message = str(e)
            is_rate_limit = "rate_limit" in message or "429" in message
            if not is_rate_limit or attempt == max_attempts - 1:
                raise
            tried_models.add(MODEL_ID)
            if switch_to_next_model(exclude=tried_models) is None:
                raise


def interrogator_chat(message, history):
    try:
        return _interrogator_chat_inner(message, history)
    except Exception as e:
        print(f"[stage error] {type(e).__name__}: {e}")
        return "The precinct's lines are jammed right now. Give it a minute and try again."


def _interrogator_chat_inner(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn.get("role"), "content": extract_text(turn.get("content"))})
        else:
            user_msg, bot_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": extract_text(user_msg)})
            if bot_msg:
                messages.append({"role": "assistant", "content": extract_text(bot_msg)})

    # The status note is folded into the user turn as plain prose rather than sent
    # as a second system message, since some models misread an extra system-role
    # message as a tool invocation.
    if case_already_closed(history):
        status_note = (
            "(Detective's internal note: the previous case is already closed. "
            "If this new message presents a genuinely new, different problem to "
            "investigate, open a brand-new case and start from scratch. But phrases "
            "like 'thanks', 'okay', 'got it', or 'okay I will do that' are just "
            "acknowledgments or confirmations, NOT new cases, even though they "
            "sound like agreement or future action. For those, reply briefly and "
            "naturally in character, WITHOUT opening a case or repeating the old "
            "closing report. If unsure, default to treating it as an acknowledgment.)\n\n"
        )
    else:
        questions_asked = count_questions_asked(history)
        remaining = max(MAX_QUESTIONS - questions_asked, 0)
        if remaining == 0:
            status_note = (
                f"(Detective's internal note: you have already asked {questions_asked} questions, "
                f"the maximum allowed. You must close the case now instead of asking another question.)\n\n"
            )
        else:
            status_note = (
                f"(Detective's internal note: you have asked {questions_asked} question so far, "
                f"you may ask up to {remaining} more if needed, or close the case now if you're confident.)\n\n"
            )

    messages.append({"role": "user", "content": status_note + extract_text(message)})

    extra_kwargs = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT else {}
    response = create_with_retry(
        model=MODEL_ID,
        messages=messages,
        max_tokens=900,
        temperature=0.4,
        **extra_kwargs,
    )
    return response.choices[0].message.content


demo = gr.ChatInterface(
    fn=interrogator_chat,
    title="The Interrogator",
    description="Present your case. The detective will investigate before concluding.",
)

# demo.launch(share=True, debug=True)  # launched in the Dynamic UI section 

# **Prompt Chaining**

In [ ]:
def clean_stage_output(text):
    """Strips a leading echoed header (e.g. the model repeating 'EVIDENCE:' at the
    start of its own answer) so the final message doesn't end up with a doubled header."""
    text = text.strip()
    known_headers = ["EVIDENCE:", "HYPOTHESES:", "MISSING:", "QUESTION:", "CONCLUSION:", "DECISION:"]
    for header in known_headers:
        if text.upper().startswith(header):
            text = text[len(header):].lstrip(" \n")
            break
    return text


def call_stage(user_content, max_tokens=200):
    """One link in the prompt chain: a single-purpose call with no conversation
    history, just the specific context that stage needs."""
    extra_kwargs = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT else {}
    response = create_with_retry(
        model=MODEL_ID,
        messages=[{"role": "user", "content": user_content}],
        max_tokens=max_tokens,
        temperature=0.4,
        **extra_kwargs,
    )
    return response.choices[0].message.content.strip()


def build_convo_text(context_turns):
    """Joins a list of (role, text) tuples into plain 'Role: text' lines. The
    caller decides what context_turns contains — see chained_interrogator_chat."""
    return "\n".join(f"{role.capitalize()}: {text}" for role, text in context_turns)


# --- Prompt-building helpers shared by the chain stages ---
USE_FEW_SHOT = True 

PERSON_RULE = (
    "Address the person you are talking to as 'you' (never 'the user' or 'the speaker'), but only "
    "for facts about them. Keep any other person or thing mentioned (a victim, a friend, a mother, "
    "a device) in the third person and never turn them into 'you'."
)


def with_examples(instruction, examples):
    """Builds a stage prompt prefix. With USE_FEW_SHOT on, the stage's worked example(s)
    follow the instruction; with it off, the same instruction runs zero-shot."""
    if not USE_FEW_SHOT:
        return instruction + "\n\n"
    return (
        instruction
        + " Match the style of the example(s) below, but never reuse their content.\n\n"
        + examples
        + "\n\nNOW THE REAL CASE\n"
    )


# --- Stage 1: Evidence Extraction ---
def extract_evidence(context_turns, message):
    """context_turns: the relevant prior turns for the CURRENT case only. Pass an
    empty list for a brand-new case, or history_since_last_close(history) for a
    continuing case, so an unrelated old case can't leak in as evidence."""
    convo_text = build_convo_text(context_turns)
    prompt = (
        "You are the EVIDENCE-EXTRACTION stage of a detective's case-analysis pipeline.\n"
        "Read the ENTIRE conversation so far, plus the newest message. Produce ONE cumulative "
        "list of ALL explicit, concrete facts stated about the situation across the WHOLE case "
        "so far — combine facts from earlier turns with anything new in this message. Do NOT "
        "drop or forget facts from earlier turns just because the newest message doesn't repeat "
        "them. No hypotheses, no psychological speculation about the person's mental or emotional "
        "state, no missing-information notes. " + PERSON_RULE + " Keep each bullet ONE short "
        "sentence, maximum 6 bullets. Do NOT include the word 'EVIDENCE' as a header in your "
        "answer — start directly with the first bullet.\n\n"
        f"CONVERSATION SO FAR:\n{convo_text if convo_text else '(none yet)'}\n\n"
        f"NEWEST MESSAGE: {message}\n\nEVIDENCE:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=280))


# --- Stage 2: Hypothesis Generation ---
# Few-shot examples for the chain stages are kept compact to limit per-turn token cost.
HYPOTHESIS_EXAMPLES = """EXAMPLE 1
EVIDENCE:
- Your laptop is slow when several programs run at once.
- Task Manager shows the CPU at 100%.
HYPOTHESES:
- A background process (an update or antivirus scan) is consuming the CPU.
- Thermal throttling from dust or a failing fan is slowing the CPU.
- One resource-heavy application is starving the others.

EXAMPLE 2 (the evidence itself states a feeling, so a hypothesis may address it)
EVIDENCE:
- You feel anxious before presentations.
- Your next presentation is on Friday.
HYPOTHESES:
- Limited rehearsal time is making Friday feel riskier.
- Fear of being judged is the main source of the anxiety.
- A bad experience at a past presentation is shaping your expectations.

EXAMPLE 3 (the evidence is thin, so the hypotheses stay general instead of inventing details)
EVIDENCE:
- Your car makes a strange noise.
HYPOTHESES:
- The noise comes from the engine or exhaust.
- The noise comes from the brakes or suspension.
- The noise depends on speed or road conditions."""


def generate_hypotheses(evidence_text):
    instruction = (
        "You are the HYPOTHESIS-GENERATION stage of a detective's case-analysis pipeline. "
        "Given ONLY this evidence, list 2-4 plausible hypotheses about the practical "
        "SITUATION or PROBLEM described, as specific as the evidence allows. Stay within what the evidence supports: never invent "
        "circumstances, people, or events it does not mention, and when the evidence is thin keep "
        "the hypotheses general rather than inventing specifics. Only address the person's feelings "
        "if the evidence itself states a feeling. " + PERSON_RULE + " Keep each bullet ONE short "
        "sentence. Bulleted list only, no other commentary."
    )
    prompt = with_examples(instruction, HYPOTHESIS_EXAMPLES) + f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:"
    return clean_stage_output(call_stage(prompt, max_tokens=280))


# --- Stage 3: Evidence Gap Detection ---
def find_missing(evidence_text, hypotheses_text):
    prompt = (
        "You are the EVIDENCE-GAP stage of a detective's case-analysis pipeline.\n"
        "Given this evidence and these hypotheses, identify the 1-3 most useful pieces of "
        "missing information that would help distinguish between the hypotheses. "
        + PERSON_RULE + " Keep each bullet ONE short sentence, no sub-explanations. Bulleted "
        "list only.\n\n"
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\nMISSING:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=220))


# --- Stage 4: Decision (close vs. continue) ---
def decide_close(evidence_text, hypotheses_text, missing_text, questions_asked, max_questions, stall_count=0):
    stall_note = ""
    if stall_count >= 2:
        stall_note = (
            f"\nNote: the last {stall_count} consecutive answers gave no new usable information. "
            "That gap is not being filled by asking again \u2014 strongly favor CLOSE with the best "
            "conclusion the evidence actually supports, rather than continuing to ask.\n"
        )
    prompt = (
        "You are the DECISION stage of a detective's case-analysis pipeline.\n"
        f"So far {questions_asked} follow-up question(s) have been asked (max allowed: {max_questions})."
        f"{stall_note}\n"
        "Given the evidence, hypotheses, and missing information below, decide: is there already "
        "enough evidence to confidently close the case, or is one more focused question needed?\n"
        "Answer with EXACTLY one word: CLOSE or CONTINUE.\n\n"
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\nMISSING:\n{missing_text}\n\nDECISION:"
    )
    decision = call_stage(prompt, max_tokens=5).strip().upper()
    return "CLOSE" if "CLOSE" in decision else "CONTINUE"


# --- Stage 5a: Next Question ---
QUESTION_EXAMPLE = """EXAMPLE
EVIDENCE:
- Your laptop is slow when several programs run at once.
- Task Manager shows the CPU at 100%.
HYPOTHESES:
- A background process is consuming the CPU.
- Thermal throttling is slowing the CPU.
MISSING:
- Which process is using the most CPU.
- Whether the slowdown continues after a restart.
QUESTION:
When the CPU is at 100%, which process is at the top of the Task Manager list?"""


def generate_question(evidence_text, hypotheses_text, missing_text, asked_questions=None):
    asked_questions = asked_questions or []
    already_asked_note = ""
    if asked_questions:
        asked_list = "\n".join(f"- {q}" for q in asked_questions)
        already_asked_note = (
            "\n\nQuestions already asked this case (do NOT repeat any of these, even reworded "
            "close to identically). If the same gap is still unfilled after being asked once, "
            "either approach it from a genuinely different angle, or if you cannot think of one, "
            "pick the next most useful DIFFERENT gap instead:\n" + asked_list
        )
    instruction = (
        "You are the NEXT-QUESTION stage of a detective's case-analysis pipeline. Given the "
        "evidence, hypotheses, and missing information below, write exactly ONE focused, "
        "high-value follow-up question about the single most useful gap. It must be one "
        "question, not two questions joined with 'and'. " + PERSON_RULE + " Return ONLY the "
        "question, nothing else."
    )
    prompt = with_examples(instruction, QUESTION_EXAMPLE) + (
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\n"
        f"MISSING:\n{missing_text}{already_asked_note}\n\nQUESTION:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=100))


# --- Stage 5b: Case Conclusion ---
CONCLUSION_EXAMPLE = """EXAMPLE
EVIDENCE:
- Your laptop is slow when several programs run at once.
- The CPU is at 100% and VS Code is the top process.
- The CPU stays high after closing VS Code until you restart.
HYPOTHESES:
- A VS Code extension is leaking CPU.
- Thermal throttling is slowing the CPU.
- Malware is consuming resources.
CONCLUSION:
PRIMARY CONCLUSION
A lingering VS Code background process, most likely an extension, is holding the CPU.

EVIDENCE
The CPU stays high after VS Code is closed and only a restart clears it, which points to a process that is not terminating rather than a hardware fault.

HYPOTHESES RULED OUT
Thermal throttling is unlikely because a restart fixes the slowdown. Malware is less likely because the problem tracks VS Code specifically. Neither was tested directly.

RECOMMENDATION
Start VS Code in safe mode and re-enable extensions one at a time to find the culprit. End any leftover language-server processes in Task Manager."""


def generate_conclusion(evidence_text, hypotheses_text):
    instruction = (
        "You are the CASE-CONCLUSION stage of a detective's case-analysis pipeline. Write a final "
        "case conclusion with these EXACT section headers: PRIMARY CONCLUSION, EVIDENCE, "
        "HYPOTHESES RULED OUT, RECOMMENDATION. Rules: (1) cite ONLY facts that appear in the "
        "EVIDENCE list; (2) never claim a symptom or circumstance is absent unless the evidence "
        "explicitly says so; (3) call a hypothesis 'ruled out' only if a stated fact directly "
        "contradicts it, otherwise call it unlikely or unconfirmed; (4) do not add mechanisms or "
        "events the evidence does not state; (5) keep it tight, a few sentences per section. "
        + PERSON_RULE
    )
    prompt = with_examples(instruction, CONCLUSION_EXAMPLE) + (
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\nCONCLUSION:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=450))


# --- Post-closure small talk: kept as a single lightweight call, not the full chain ---
def classify_after_close(message):
    # Short, common acknowledgments are classified directly, without an LLM call,
    # since a model classifier is least reliable on very short, ambiguous input.
    trivial_acks = {
        "ok", "okay", "k", "kk", "alright", "sure", "thanks", "thank you", "cool",
        "got it", "sounds good", "noted", "fine", "yep", "yup", "great", "nice",
    }
    if message.strip().lower().rstrip("!.") in trivial_acks:
        return "ACK"

    prompt = (
        "The previous case was already closed. Classify this new user message as EXACTLY one word:\n"
        "ACK — just an acknowledgment, thanks, or confirmation like 'okay', 'thanks', 'got it', "
        "'okay i will do that', 'sure' (even if it sounds like future action, it is NOT a new case).\n"
        "NEW_CASE — a genuinely new, different problem to investigate.\n"
        "If unsure, default to ACK.\n\n"
        f'Message: "{message}"\n\nAnswer with exactly one word (ACK or NEW_CASE):'
    )
    answer = call_stage(prompt, max_tokens=5).strip().upper()
    return "NEW_CASE" if "NEW_CASE" in answer else "ACK"


def brief_ack_reply(message):
    prompt = (
        "You are THE INTERROGATOR, a sharp detective persona, speaking casually now that the case "
        "is closed. The user just sent a brief acknowledgment or thanks. Reply briefly (1-2 sentences), "
        "in character, without opening a new case or repeating any case report.\n\n"
        f'User said: "{message}"\n\nYour brief in-character reply:'
    )
    return call_stage(prompt, max_tokens=80)


def chained_interrogator_chat(message, history):
    try:
        return _chained_interrogator_chat_inner(message, history)
    except Exception as e:
        print(f"[stage error] {type(e).__name__}: {e}")
        return "The precinct's lines are jammed right now. Give it a minute and try again."


def _chained_interrogator_chat_inner(message, history):
    message = extract_text(message)

    if case_already_closed(history):
        if classify_after_close(message) == "ACK":
            return brief_ack_reply(message)
        # Otherwise: a genuinely new case. Fall through and run the chain as CASE OPENED.

    is_new_case = (len(history) == 0) or case_already_closed(history)

    # A brand-new case starts with no prior context at all, not even "since the last
    # close" — that window can still contain the small-talk exchange that followed
    # the old case's closure, which is not evidence for the new case.
    evidence_context = [] if is_new_case else history_since_last_close(history)

    if is_new_case:
        print("=== NEW CASE ===\n")

    print("--- Stage 1: Evidence Extraction ---")
    evidence = extract_evidence(evidence_context, message)
    print(evidence)

    print("\n--- Stage 2: Hypothesis Generation ---")
    hypotheses = generate_hypotheses(evidence)
    print(hypotheses)

    print("\n--- Stage 3: Evidence Gap ---")
    missing = find_missing(evidence, hypotheses)
    print(missing)

    questions_asked = count_questions_asked(history)
    remaining = max(MAX_QUESTIONS - questions_asked, 0)
    decision = "CLOSE" if remaining == 0 else decide_close(evidence, hypotheses, missing, questions_asked, MAX_QUESTIONS)
    print(f"\n--- Stage 4: Decision --- {decision}")

    if decision == "CLOSE":
        print("\n--- Stage 5b: Case Conclusion ---")
        conclusion = generate_conclusion(evidence, hypotheses)
        print(conclusion)
        return f"CASE CLOSED\n\n{conclusion}"
    else:
        print("\n--- Stage 5a: Next Question ---")
        question = generate_question(evidence, hypotheses, missing)
        print(question)
        header = "CASE OPENED" if is_new_case else "CASE UPDATE"
        return (
            f"{header}\n\n"
            f"EVIDENCE:\n{evidence}\n\n"
            f"HYPOTHESES:\n{hypotheses}\n\n"
            f"MISSING:\n{missing}\n\n"
            f"QUESTION:\n{question}"
        )


demo = gr.ChatInterface(
    fn=chained_interrogator_chat,
    title="The Interrogator — Prompt-Chained",
    description="Present your case. Each response is built from 4-6 separate staged reasoning calls — check the cell's log output below to see each stage.",
)

# demo.launch(share=True, debug=True)  # launched in the Dynamic UI section 

# **Structured Case State**

In [ ]:
def new_case_state():
    """A fresh, empty case. Evidence accumulates into this via list.append() as the
    case progresses, instead of re-deriving the whole list from scratch each turn."""
    return {
        "status": "OPEN",
        "evidence": [],
        "hypotheses": "",
        "missing": "",
        "questions_asked": 0,
        "last_question": "",
    }


def parse_bullets(text):
    """Splits a bulleted LLM response into a clean list of strings, dropping
    bullet markers, blank lines, and non-fact filler (e.g. the model writing
    "no new facts" as a sentence instead of returning nothing)."""
    junk_phrases = ("no new", "none", "nothing new", "no concrete fact", "no additional fact")
    lines = []
    for line in text.split("\n"):
        line = line.strip().lstrip("*-•").strip()
        if line and not any(line.lower().startswith(p) for p in junk_phrases):
            lines.append(line)
    return lines


def extract_new_evidence(message, last_question="", known_facts=None):
    """Extracts only the new facts from this single message, not the case's full
    cumulative evidence. Takes the last question asked as context, so a short or
    referential answer (e.g. "yes", "I was late") can be correctly interpreted
    instead of producing a vague, disconnected fact."""
    question_context = f'The detective just asked: "{last_question}"\n' if last_question else ""
    known_context = ""
    if known_facts:
        known_context = "ALREADY KNOWN (do not repeat or rephrase these):\n" + "\n".join(f"- {f}" for f in known_facts) + "\n\n"
    prompt = (
        "Extract ONLY the new explicit, concrete facts stated in this single message — "
        "no hypotheses, no psychological speculation, no missing-info notes. " + PERSON_RULE + " "
        "One short sentence per bullet, maximum 3 bullets. Use the question "
        "context below, if given, to interpret short or referential answers correctly — e.g. "
        "if asked about a time limit and the answer is 'yes', state what was confirmed, not "
        "just 'you confirmed the action'. If the message has no new concrete facts (e.g. it's "
        "vague or just an exclamation), return nothing.\n\n"
        f"{known_context}{question_context}"
        f"MESSAGE: {message}\n\nNEW FACTS (bulleted, or leave blank if none):"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=120))


def stateful_investigate(message, case_state):
    """Public entry point: catches any stage failure (e.g. an API rate limit
    mid-investigation) and degrades gracefully instead of crashing the app.
    case_state is left untouched on failure, so retrying continues where it left off."""
    try:
        return _stateful_investigate_inner(message, case_state)
    except Exception as e:
        print(f"[stage error] {type(e).__name__}: {e}")
        friendly = (
            "The precinct's lines are jammed right now. Give it a minute and send "
            "that again; nothing about your case has been lost."
        )
        return friendly, case_state


def _stateful_investigate_inner(message, case_state):
    """Reads and writes an explicit case_state dict instead of re-inferring case
    status, evidence, and question count from raw chat history each turn."""
    message = extract_text(message)

    if case_state["status"] == "CLOSED":
        if classify_after_close(message) == "ACK":
            return brief_ack_reply(message), case_state
        case_state = new_case_state()

    is_new_case = (len(case_state["evidence"]) == 0)


    if is_new_case:
        print("=== NEW CASE ===\n")
    print("--- Stage 1: New-Evidence Extraction ---")
    new_facts_text = extract_new_evidence(message, case_state.get("last_question", ""), case_state["evidence"])
    new_facts = parse_bullets(new_facts_text)
    case_state["evidence"].extend(new_facts)
    print(new_facts if new_facts else "(no new concrete facts)")

    evidence_text = "\n".join(f"- {e}" for e in case_state["evidence"]) or "(none yet)"

    print("\n--- Stage 2: Hypothesis Generation ---")
    case_state["hypotheses"] = generate_hypotheses(evidence_text)
    print(case_state["hypotheses"])

    print("\n--- Stage 3: Evidence Gap ---")
    case_state["missing"] = find_missing(evidence_text, case_state["hypotheses"])
    print(case_state["missing"])

    remaining = max(MAX_QUESTIONS - case_state["questions_asked"], 0)
    decision = "CLOSE" if remaining == 0 else decide_close(
        evidence_text, case_state["hypotheses"], case_state["missing"],
        case_state["questions_asked"], MAX_QUESTIONS,
    )
    print(f"\n--- Stage 4: Decision --- {decision}")

    if decision == "CLOSE":
        print("\n--- Stage 5b: Case Conclusion ---")
        conclusion = generate_conclusion(evidence_text, case_state["hypotheses"])
        print(conclusion)
        case_state["status"] = "CLOSED"
        reply = f"CASE CLOSED\n\n{conclusion}"
    else:
        print("\n--- Stage 5a: Next Question ---")
        question = generate_question(evidence_text, case_state["hypotheses"], case_state["missing"])
        print(question)
        case_state["questions_asked"] += 1
        case_state["last_question"] = question
        header = "CASE OPENED" if is_new_case else "CASE UPDATE"
        reply = (
            f"{header}\n\n"
            f"EVIDENCE:\n{evidence_text}\n\n"
            f"HYPOTHESES:\n{case_state['hypotheses']}\n\n"
            f"MISSING:\n{case_state['missing']}\n\n"
            f"QUESTION:\n{question}"
        )

    return reply, case_state


# gr.Blocks + gr.State carries case_state across turns, which gr.ChatInterface
# (used in the previous two sections) does not support.
with gr.Blocks(title="The Interrogator — Structured Case State") as demo:
    gr.Markdown("### The Interrogator — Structured Case State\nPresent your case. Case status, evidence, and question count are tracked explicitly, not re-inferred from chat text each turn.")
    chatbot = gr.Chatbot(height=450)
    case_state_box = gr.State(new_case_state())
    msg = gr.Textbox(label="Your message", placeholder="Present your case...")
    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        new_case_btn = gr.Button("New Case")

    def handle_send(message, chat_history, case_state):
        if not message or not message.strip():
            return chat_history, case_state, ""
        reply, case_state = stateful_investigate(message, case_state)
        chat_history = chat_history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply},
        ]
        return chat_history, case_state, ""

    def handle_new_case(chat_history, case_state):
        """Clears the visible chat as well as the internal state, so there is a
        clear line between the old case and the new one."""
        fresh_chat = [{"role": "assistant", "content": "New case started. What's going on?"}]
        return fresh_chat, new_case_state()

    msg.submit(handle_send, [msg, chatbot, case_state_box], [chatbot, case_state_box, msg])
    send_btn.click(handle_send, [msg, chatbot, case_state_box], [chatbot, case_state_box, msg])
    new_case_btn.click(handle_new_case, [chatbot, case_state_box], [chatbot, case_state_box])

# demo.launch(share=True, debug=True)  # launched in the Dynamic UI section 

# **Context Handling**

In [ ]:
def get_context_window(model_id):
    """Reads the real context window for this model from Groq's own model list
    (each model object includes a context_window field), instead of hardcoding a
    number that could go stale if the auto-selected model changes."""
    try:
        for m in client.models.list().data:
            if m.id == model_id:
                return getattr(m, "context_window", None)
    except Exception:
        pass
    return None


CONTEXT_WINDOW = get_context_window(MODEL_ID)
print(f"Model in use: {MODEL_ID}")
print(f"Context window: {CONTEXT_WINDOW if CONTEXT_WINDOW else 'not reported by the API'} tokens")


def estimate_tokens(text):
    """Rough character-based estimate (~4 characters per token), used only to decide
    when to trim, not for billing accuracy. Good enough for a retention policy: being
    off by 20% still trims at roughly the right point."""
    return max(1, len(text) // 4)


# Conservative working budget, well under the real context window, so the retention
# policy engages long before the model would actually refuse the request.
CONTEXT_BUDGET_TOKENS = min(6000, (CONTEXT_WINDOW or 8192) // 2)
RECENT_TURNS_TO_KEEP = 6  # most recent user+assistant messages always kept verbatim


def summarize_old_turns(old_turns):
    """Collapses older conversation turns into one dense paragraph of established
    facts, using the model itself as the summarizer. old_turns: list of {'role','content'}."""
    convo_text = "\n".join(f"{t['role'].capitalize()}: {t['content']}" for t in old_turns)
    prompt = (
        "Summarize the following portion of a detective-style case conversation into a "
        "short, dense paragraph (3-5 sentences). Capture only the concrete facts already "
        "established — no hypotheses, no meta-commentary about the conversation itself. "
        "Address the person as 'you'.\n\n"
        f"{convo_text}\n\nSUMMARY:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=200))


def manage_context(history, system_prompt):
    """Recent-history retention with summarization: keeps the most recent
    RECENT_TURNS_TO_KEEP turns verbatim always. Once the estimated token count of
    (system prompt + full history) crosses CONTEXT_BUDGET_TOKENS, everything older
    than the recent window is replaced with a single summary turn. Below budget, or
    on a short conversation, history passes through unchanged."""
    normalized = normalize_history(history)
    total_tokens = estimate_tokens(system_prompt) + sum(estimate_tokens(t) for _, t in normalized)

    if total_tokens <= CONTEXT_BUDGET_TOKENS or len(normalized) <= RECENT_TURNS_TO_KEEP:
        return history, False  # False: no trimming happened this turn

    recent = normalized[-RECENT_TURNS_TO_KEEP:]
    older = normalized[:-RECENT_TURNS_TO_KEEP]
    summary = summarize_old_turns([{"role": r, "content": t} for r, t in older])
    summarized_turn = {"role": "user", "content": f"(Case summary of earlier turns: {summary})"}
    trimmed = [summarized_turn] + [{"role": r, "content": t} for r, t in recent]
    return trimmed, True




In [ ]:
def _interrogator_chat_inner(message, history):
    """Redefines the Persona and Few-Shot version to apply manage_context before
    building the message list, so a long conversation gets summarized instead of
    growing the request without bound."""
    history, was_trimmed = manage_context(history, SYSTEM_PROMPT)
    if was_trimmed:
        print(f"[context] trimmed history to a summary + last {RECENT_TURNS_TO_KEEP} turns")

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn.get("role"), "content": extract_text(turn.get("content"))})
        else:
            user_msg, bot_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": extract_text(user_msg)})
            if bot_msg:
                messages.append({"role": "assistant", "content": extract_text(bot_msg)})

    if case_already_closed(history):
        status_note = (
            "(Detective's internal note: the previous case is already closed. "
            "If this new message presents a genuinely new, different problem to "
            "investigate, open a brand-new case and start from scratch. But phrases "
            "like 'thanks', 'okay', 'got it', or 'okay I will do that' are just "
            "acknowledgments or confirmations, NOT new cases, even though they "
            "sound like agreement or future action. For those, reply briefly and "
            "naturally in character, WITHOUT opening a case or repeating the old "
            "closing report. If unsure, default to treating it as an acknowledgment.)\n\n"
        )
    else:
        questions_asked = count_questions_asked(history)
        remaining = max(MAX_QUESTIONS - questions_asked, 0)
        if remaining == 0:
            status_note = (
                f"(Detective's internal note: you have already asked {questions_asked} questions, "
                f"the maximum allowed. You must close the case now instead of asking another question.)\n\n"
            )
        else:
            status_note = (
                f"(Detective's internal note: you have asked {questions_asked} question so far, "
                f"you may ask up to {remaining} more if needed, or close the case now if you're confident.)\n\n"
            )

    messages.append({"role": "user", "content": status_note + extract_text(message)})

    extra_kwargs = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT else {}
    response = create_with_retry(
        model=MODEL_ID,
        messages=messages,
        max_tokens=900,
        temperature=0.4,
        **extra_kwargs,
    )
    return response.choices[0].message.content




In [ ]:
def build_synthetic_long_conversation(n_turns=20):
    """A synthetic conversation long enough to demonstrate the retention policy
    without spending a real multi-turn session's worth of tokens getting there."""
    facts = [
        "My head has been hurting since this morning.",
        "It's mostly in my forehead and behind my eyes.",
        "I haven't had any water since breakfast.",
        "I've been staring at my laptop for about five hours straight.",
        "I did take a short walk around noon.",
        "The pain gets a bit worse when I look at bright light.",
        "I haven't eaten lunch yet either.",
        "I slept about five hours last night.",
        "I don't have a fever as far as I can tell.",
        "This has happened a couple of times before during exam weeks.",
    ]
    history = []
    for i in range(n_turns):
        fact = facts[i % len(facts)]
        history.append({"role": "user", "content": fact})
        history.append({
            "role": "assistant",
            "content": (
                "CASE UPDATE\n\nEVIDENCE:\n- " + fact + "\n\nHYPOTHESES:\n"
                "- Tension headache from prolonged screen time.\n"
                "- Dehydration contributing to the pain.\n\nMISSING:\n"
                "- Whether the pain is throbbing or constant.\n\nQUESTION:\n"
                "Is the pain throbbing or a constant pressure?"
            ),
        })
    return history


demo_history = build_synthetic_long_conversation(n_turns=70)  
before_tokens = estimate_tokens(SYSTEM_PROMPT) + sum(
    estimate_tokens(extract_text(t["content"])) for t in demo_history
)
print(f"Before: {len(demo_history)} turns, ~{before_tokens} estimated tokens (budget: {CONTEXT_BUDGET_TOKENS})")

trimmed_history, was_trimmed = manage_context(demo_history, SYSTEM_PROMPT)
after_tokens = estimate_tokens(SYSTEM_PROMPT) + sum(
    estimate_tokens(t["content"] if isinstance(t, dict) else extract_text(t)) for t in trimmed_history
)
print(f"After:  {len(trimmed_history)} turns, ~{after_tokens} estimated tokens, trimmed={was_trimmed}")

if was_trimmed:
    print("\n--- Summary turn that replaced the older messages ---")
    print(trimmed_history[0]["content"])


# **Reasoning Mode**

In [ ]:
def generate_conclusion_with_reasoning(evidence_text, hypotheses_text):
    """Chain-of-thought version of generate_conclusion: elicits an explicit,
    structured reasoning trace under INVESTIGATION SUMMARY before the same
    four-section FINAL CASE CONCLUSION format. Same grounding rules as the
    non-reasoning version; the only difference is showing the work first."""
    instruction = (
        "You are the CASE-CONCLUSION stage of a detective's case-analysis pipeline, "
        "running in REASONING MODE. Think through this step by step BEFORE answering. "
        "Be concise throughout \u2014 this is a brief investigation trace, not an essay. "
        "Do not use markdown bold headers, numbered sub-lists, or restate the evidence "
        "verbatim anywhere below.\n\n"
        "First, under the header INVESTIGATION SUMMARY, write EXACTLY these four lines. "
        "Each line has a HARD LIMIT of 20 words \u2014 count as you write and cut anything over. "
        "A short sentence with one clause, not a compound sentence stacking multiple clauses:\n"
        "Evidence considered: a single short clause naming only the most relevant facts.\n"
        "Hypotheses considered: a single short clause naming only the most relevant hypotheses.\n"
        "Key missing or decisive evidence: a single short clause on the one fact that settles it.\n"
        "Decision basis: AT MOST 4 numbered steps, each under 15 words, one plain clause each "
        "(include any arithmetic, but no sub-bullets, no extended discussion, no restating lists).\n\n"
        "Then, under the header FINAL CASE CONCLUSION, write these EXACT section headers: "
        "PRIMARY CONCLUSION, EVIDENCE, HYPOTHESES RULED OUT, RECOMMENDATION. The PRIMARY "
        "CONCLUSION must directly follow from the deduction you just showed, and must give a "
        "concrete answer or recommendation \u2014 do not end the case still listing open "
        "questions. Rules: (1) cite ONLY facts that appear in the EVIDENCE list; (2) never "
        "claim a symptom or circumstance is absent unless the evidence explicitly says so; "
        "(3) call a hypothesis 'ruled out' only if a stated fact directly contradicts it, "
        "otherwise call it unlikely or unconfirmed; (4) do not add mechanisms or events the "
        "evidence does not state; (5) keep the FINAL CASE CONCLUSION tight, a few sentences "
        "per section. "
        + PERSON_RULE
    )
    prompt = instruction + f"\n\nEVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\nINVESTIGATION SUMMARY:"
    return clean_stage_output(call_stage(prompt, max_tokens=1100))


def split_reasoning_and_conclusion(text):
    """Splits a generate_conclusion_with_reasoning output into (investigation_summary,
    final_conclusion). final_conclusion keeps the same PRIMARY CONCLUSION/EVIDENCE/
    HYPOTHESES RULED OUT/RECOMMENDATION shape generate_conclusion always produces, so
    existing recommendation-extraction code works unchanged on it."""
    marker = "FINAL CASE CONCLUSION"
    idx = text.upper().find(marker)
    if idx == -1:
        conclusion = (
            "PRIMARY CONCLUSION\n(The reasoning trace ran long and was cut off before "
            "reaching a final conclusion. See the investigation summary above for the "
            "partial reasoning; try again for a complete answer.)\n\nEVIDENCE\n\n"
            "HYPOTHESES RULED OUT\n\nRECOMMENDATION\n"
        )
        return text.strip(), conclusion
    summary = text[:idx].strip()
    conclusion = text[idx + len(marker):].strip(" :\n")
    return summary, conclusion


# --- Demonstration: two reasoning-sensitive prompts, with vs. without reasoning mode ---
DEMO_PROBLEMS = {
    "bat-and-ball": {
        "evidence": (
            "- A bat and a ball cost $1.10 in total.\n"
            "- The bat costs $1.00 more than the ball."
        ),
        "hypotheses": "- This is a self-contained arithmetic word problem to solve directly from the evidence.",
        "correct_answer": "The ball costs $0.05 (bat = $1.05, ball = $0.05; sum = $1.10, difference = $1.00).",
    },
    "three-boxes": {
        "evidence": (
            "- There are three boxes labeled Apples, Oranges, and Apples+Oranges.\n"
            "- All three labels are known to be wrong.\n"
            "- One fruit was drawn from the box labeled Apples+Oranges, and it was an apple."
        ),
        "hypotheses": "- This is a self-contained logic puzzle to solve directly from the evidence.",
        "correct_answer": (
            "Box labeled Apples+Oranges is actually all apples. Box labeled Oranges is "
            "actually the mixed (Apples+Oranges) box. Box labeled Apples is actually all oranges."
        ),
    },
}

for name, problem in DEMO_PROBLEMS.items():
    print(f"\n{'=' * 70}\n{name.upper()}\nCorrect answer: {problem['correct_answer']}\n{'=' * 70}")

    print("\n--- Reasoning Mode OFF (generate_conclusion) ---")
    off_result = generate_conclusion(problem["evidence"], problem["hypotheses"])
    print(off_result)

    print("\n--- Reasoning Mode ON (generate_conclusion_with_reasoning) ---")
    on_result = generate_conclusion_with_reasoning(problem["evidence"], problem["hypotheses"])
    summary, final = split_reasoning_and_conclusion(on_result)
    print("[Investigation Summary]")
    print(summary)
    print("\n[Final Case Conclusion]")
    print(final)


# **Dynamic UI**

In [ ]:
_case_counter = {"n": 0}


def new_case_state():
    """A fresh, empty case. Evidence accumulates into this via list.append() as the
    case progresses, instead of re-deriving the whole list from scratch each turn.
    Redefines the Structured Case State version to add case_id, stage_label, and
    recommendation, used by the panel below to display status without re-deriving it."""
    _case_counter["n"] += 1
    return {
        "case_id": _case_counter["n"],
        "status": "OPEN",
        "stage_label": "CASE OPENED",
        "evidence": [],
        "hypotheses": "",
        "missing": "",
        "recommendation": "",
        "reasoning_trace": "",
        "questions_asked": 0,
        "last_question": "",
        "asked_questions": [],  # full history, so generate_question can't repeat itself
        "stall_count": 0,  # consecutive turns with no new evidence extracted
    }


EVIDENCE_RETENTION_LIMIT = 10  # most recent facts always kept verbatim
EVIDENCE_TOKEN_BUDGET = 500  # older facts are summarized once the evidence list's estimated tokens cross this


def summarize_evidence_facts(facts):
    """Collapses older evidence bullets into one dense sentence that preserves every
    concrete detail. This is the same idea as summarize_old_turns (Context Handling),
    adapted to the structured evidence list instead of raw chat turns, since this UI
    never sends raw conversation history to the model at all."""
    joined = "\n".join(f"- {f}" for f in facts)
    prompt = (
        "Combine the following case facts into ONE dense sentence that preserves every "
        "concrete detail. No hypotheses, no commentary. Address the person as 'you'.\n\n"
        f"{joined}\n\nCOMBINED FACT:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=150))


def manage_evidence(case_state):
    """Recent-history retention applied to the structured evidence list: the most
    recent EVIDENCE_RETENTION_LIMIT facts always stay verbatim. Once the list's
    estimated token count crosses EVIDENCE_TOKEN_BUDGET, everything older than that
    window is collapsed into one combined fact. Under MAX_QUESTIONS's default of 5,
    a single case rarely produces enough facts to trigger this - it exists as a real,
    wired-in safety net for longer cases (e.g. if MAX_QUESTIONS is raised later),
    not something the default settings are expected to hit often."""
    facts = case_state["evidence"]
    total_tokens = sum(estimate_tokens(f) for f in facts)
    if len(facts) <= EVIDENCE_RETENTION_LIMIT or total_tokens <= EVIDENCE_TOKEN_BUDGET:
        return False
    recent = facts[-EVIDENCE_RETENTION_LIMIT:]
    older = facts[:-EVIDENCE_RETENTION_LIMIT]
    combined = summarize_evidence_facts(older)
    case_state["evidence"] = [f"(Earlier facts combined: {combined})"] + recent
    return True


def _stateful_investigate_inner(message, case_state, reasoning_mode=False):
    """Redefines the Structured Case State version to also set case_state['stage_label']
    (CASE OPENED / INVESTIGATING / CASE CLOSED) for display in the panel below, and to
    branch to the chain-of-thought conclusion stage when reasoning_mode is on."""
    message = extract_text(message)

    if case_state["status"] == "CLOSED":
        if classify_after_close(message) == "ACK":
            return brief_ack_reply(message), case_state
        case_state = new_case_state()

    is_new_case = (len(case_state["evidence"]) == 0)


    if is_new_case:
        print("=== NEW CASE ===\n")
    print("--- Stage 1: New-Evidence Extraction ---")
    new_facts_text = extract_new_evidence(message, case_state.get("last_question", ""), case_state["evidence"])
    new_facts = parse_bullets(new_facts_text)
    case_state["evidence"].extend(new_facts)
    print(new_facts if new_facts else "(no new concrete facts)")
    case_state["stall_count"] = 0 if new_facts else case_state.get("stall_count", 0) + 1
    if manage_evidence(case_state):
        print(f"[context] evidence trimmed to {EVIDENCE_RETENTION_LIMIT} recent facts + 1 summary")

    evidence_text = "\n".join(f"- {e}" for e in case_state["evidence"]) or "(none yet)"

    print("\n--- Stage 2: Hypothesis Generation ---")
    case_state["hypotheses"] = generate_hypotheses(evidence_text)
    print(case_state["hypotheses"])

    print("\n--- Stage 3: Evidence Gap ---")
    case_state["missing"] = find_missing(evidence_text, case_state["hypotheses"])
    print(case_state["missing"])

    case_state["stage_label"] = "CASE OPENED" if is_new_case else "INVESTIGATING"

    remaining = max(MAX_QUESTIONS - case_state["questions_asked"], 0)
    decision = "CLOSE" if remaining == 0 else decide_close(
        evidence_text, case_state["hypotheses"], case_state["missing"],
        case_state["questions_asked"], MAX_QUESTIONS, case_state.get("stall_count", 0),
    )
    print(f"\n--- Stage 4: Decision --- {decision}")

    if decision == "CLOSE":
        case_state["status"] = "CLOSED"
        case_state["stage_label"] = "CASE CLOSED"

        if reasoning_mode:
            print("\n--- Stage 5b: Case Conclusion (Reasoning Mode) ---")
            raw = generate_conclusion_with_reasoning(evidence_text, case_state["hypotheses"])
            trace, conclusion = split_reasoning_and_conclusion(raw)
            case_state["reasoning_trace"] = trace
            print("[Investigation Summary]\n" + trace)
            print("\n[Final Case Conclusion]\n" + conclusion)
            quoted_trace = "> **INVESTIGATION SUMMARY**\n> " + trace.replace("\n", "\n> ")
            reply = f"CASE CLOSED\n\n{quoted_trace}\n\n**FINAL CASE CONCLUSION**\n\n{conclusion}"
        else:
            print("\n--- Stage 5b: Case Conclusion ---")
            conclusion = generate_conclusion(evidence_text, case_state["hypotheses"])
            print(conclusion)
            case_state["reasoning_trace"] = ""
            reply = f"CASE CLOSED\n\n{conclusion}"

        idx = conclusion.upper().find("RECOMMENDATION")
        if idx != -1:
            case_state["recommendation"] = conclusion[idx + len("RECOMMENDATION"):].strip(" :\n*")
        else:
            case_state["recommendation"] = conclusion
    else:
        print("\n--- Stage 5a: Next Question ---")
        question = generate_question(
            evidence_text, case_state["hypotheses"], case_state["missing"], case_state.get("asked_questions", []),
        )
        if not question.strip():
            # Safety net
            fallback_bullets = parse_bullets(case_state["missing"])
            if fallback_bullets:
                bullet = fallback_bullets[0].rstrip(". ")
                question = bullet if bullet.endswith("?") else bullet + "?"
            else:
                question = "Can you tell me more about what's going on?"
            print(f"[fallback] generate_question returned empty; using: {question}")
        print(question)
        case_state["questions_asked"] += 1
        case_state["last_question"] = question
        case_state.setdefault("asked_questions", []).append(question)
        header = "CASE OPENED" if is_new_case else "CASE UPDATE"
        reply = (
            f"{header}\n\n"
            f"EVIDENCE:\n{evidence_text}\n\n"
            f"HYPOTHESES:\n{case_state['hypotheses']}\n\n"
            f"MISSING:\n{case_state['missing']}\n\n"
            f"QUESTION:\n{question}"
        )

    return reply, case_state


def stateful_investigate(message, case_state, reasoning_mode=False):
    """Redefines the Structured Case State version's public wrapper to also accept and
    thread through reasoning_mode. Still catches any stage failure and degrades
    gracefully instead of crashing the app; case_state is left untouched on failure."""
    try:
        return _stateful_investigate_inner(message, case_state, reasoning_mode)
    except Exception as e:
        print(f"[stage error] {type(e).__name__}: {e}")
        friendly = (
            "The precinct's lines are jammed right now. Give it a minute and send "
            "that again; nothing about your case has been lost."
        )
        return friendly, case_state


STATUS_COLORS = {
    "CASE OPENED": "#7c3aed",
    "INVESTIGATING": "#a855f7",
    "CASE CLOSED": "#4c1d95",
}


INVESTIGATING_FIGURE = """<div class="interrogator-figure"><svg viewBox="0 0 200 170" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Detective investigating the case">
  <defs>
    <radialGradient id="lampGlowOpen" cx="50%" cy="40%" r="60%">
      <stop offset="0%" stop-color="#c4b5fd" stop-opacity="0.5"/>
      <stop offset="100%" stop-color="#c4b5fd" stop-opacity="0"/>
    </radialGradient>
  </defs>
  <ellipse class="lamp-glow" cx="100" cy="88" rx="94" ry="80" fill="url(#lampGlowOpen)"/>
  <path d="M66 168 L76 98 Q100 84 124 98 L134 168 Z" fill="#3b0764" stroke="#c4b5fd" stroke-width="2"/>
  <path d="M100 92 L89 106 L100 130 L111 106 Z" fill="#1e1b4b" stroke="#a78bfa" stroke-width="1.5"/>
  <rect x="75" y="132" width="50" height="6" fill="#7c3aed"/>
  <g class="det-head">
    <circle cx="100" cy="76" r="15" fill="#4c1d95" stroke="#c4b5fd" stroke-width="2"/>
    <circle cx="95" cy="77" r="1.8" fill="#ede9fe"/>
    <circle cx="105" cy="77" r="1.8" fill="#ede9fe"/>
    <ellipse cx="100" cy="64" rx="28" ry="6" fill="#2e1065" stroke="#c4b5fd" stroke-width="2"/>
    <path d="M83 64 Q83 42 100 42 Q117 42 117 64 Z" fill="#2e1065" stroke="#c4b5fd" stroke-width="2"/>
    <rect x="83" y="56" width="34" height="5" fill="#a855f7"/>
  </g>
  <g transform="translate(124 102)">
    <g class="magnifier-arm">
      <line x1="0" y1="0" x2="26" y2="-12" stroke="#4c1d95" stroke-width="9" stroke-linecap="round"/>
      <line x1="0" y1="0" x2="26" y2="-12" stroke="#c4b5fd" stroke-width="1.5" stroke-linecap="round"/>
      <line x1="26" y1="-12" x2="38" y2="-22" stroke="#f5d0fe" stroke-width="4" stroke-linecap="round"/>
      <circle cx="46" cy="-30" r="14" fill="rgba(216,180,254,0.18)" stroke="#f5d0fe" stroke-width="3"/>
      <path d="M38 -34 Q42 -41 50 -41" fill="none" stroke="#ffffff" stroke-width="2" stroke-linecap="round" opacity="0.8"/>
    </g>
  </g>
  <text class="qmark q1" x="140" y="34" fill="#e9d5ff">?</text>
  <text class="qmark q2" x="176" y="44" fill="#e9d5ff">?</text>
  <text class="qmark q3" x="32" y="50" fill="#e9d5ff">?</text>
</svg><div class="figure-caption">Interrogation in progress</div></div>"""

CLOSED_FIGURE = """<div class="interrogator-figure"><svg viewBox="0 0 200 170" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Detective with the closed case file">
  <defs>
    <radialGradient id="lampGlowClosed" cx="50%" cy="40%" r="60%">
      <stop offset="0%" stop-color="#a78bfa" stop-opacity="0.32"/>
      <stop offset="100%" stop-color="#a78bfa" stop-opacity="0"/>
    </radialGradient>
  </defs>
  <ellipse cx="100" cy="88" rx="94" ry="80" fill="url(#lampGlowClosed)"/>
  <path d="M66 168 L76 98 Q100 84 124 98 L134 168 Z" fill="#3b0764" stroke="#c4b5fd" stroke-width="2"/>
  <path d="M100 92 L89 106 L100 122 L111 106 Z" fill="#1e1b4b" stroke="#a78bfa" stroke-width="1.5"/>
  <rect x="72" y="110" width="56" height="13" rx="6.5" fill="#4c1d95" stroke="#c4b5fd" stroke-width="2"/>
  <rect x="72" y="122" width="56" height="13" rx="6.5" fill="#3b0764" stroke="#c4b5fd" stroke-width="2"/>
  <circle cx="100" cy="76" r="15" fill="#4c1d95" stroke="#c4b5fd" stroke-width="2"/>
  <path d="M91 77 Q95 73 99 77" fill="none" stroke="#ede9fe" stroke-width="1.8" stroke-linecap="round"/>
  <path d="M101 77 Q105 73 109 77" fill="none" stroke="#ede9fe" stroke-width="1.8" stroke-linecap="round"/>
  <path d="M95 84 Q100 88 105 84" fill="none" stroke="#ede9fe" stroke-width="1.6" stroke-linecap="round"/>
  <g transform="translate(100 64)">
    <g class="hat-tip">
      <ellipse cx="0" cy="0" rx="28" ry="6" fill="#2e1065" stroke="#c4b5fd" stroke-width="2"/>
      <path d="M-17 0 Q-17 -22 0 -22 Q17 -22 17 0 Z" fill="#2e1065" stroke="#c4b5fd" stroke-width="2"/>
      <rect x="-17" y="-8" width="34" height="5" fill="#a855f7"/>
    </g>
  </g>
  <g class="folder-thump">
    <g transform="translate(160 124)">
      <path d="M-28 -24 L-10 -24 L-6 -19 L28 -19 L28 22 L-28 22 Z" fill="#ede9fe" stroke="#c4b5fd" stroke-width="2" stroke-linejoin="round"/>
      <g transform="translate(0 2)">
        <g class="stamp-slam">
          <rect x="-24" y="-9" width="48" height="18" rx="2" fill="none" stroke="#dc2626" stroke-width="2.5"/>
          <text x="0" y="4" text-anchor="middle" font-size="10" font-weight="700" fill="#dc2626" font-family="'Special Elite', monospace" letter-spacing="1">CLOSED</text>
        </g>
        <g class="impact" stroke="#dc2626" stroke-width="2" stroke-linecap="round">
          <line x1="-30" y1="-12" x2="-37" y2="-19"/>
          <line x1="30" y1="-12" x2="37" y2="-19"/>
          <line x1="0" y1="-15" x2="0" y2="-24"/>
        </g>
      </g>
    </g>
  </g>
</svg><div class="figure-caption">Case file stamped</div></div>"""


def render_status_html(case_state):
    """Status badge beside an animated detective (icon on the left, badge on the
    right): an investigating figure while the case is open, and a different one
    stamping the file once it is closed."""
    color = STATUS_COLORS.get(case_state["stage_label"], "#7c3aed")
    title = f"Case #{case_state['case_id']} — {case_state['questions_asked']} question(s) asked"
    figure = CLOSED_FIGURE if case_state["status"] == "CLOSED" else INVESTIGATING_FIGURE
    return (
        '<div class="status-row">'
        f"{figure}"
        f'<div class="status-badge" style="background:{color};">'
        f'{title}<br><span style="font-size:1.2em;">{case_state["stage_label"]}</span>'
        "</div>"
        "</div>"
    )


REASONING_TRACE_LABELS = [
    "Evidence considered",
    "Hypotheses considered",
    "Key missing or decisive evidence",
    "Decision basis",
]


def format_reasoning_trace(trace_text):
    """Reformats the raw investigation-summary text into clearly separated, labeled
    sub-sections instead of one dense paragraph, and turns the Decision basis's
    numbered steps into an actual markdown list — so it reads as cleanly as the
    Evidence (bullets) and Recommendation (short paragraph) sections beside it."""
    if not trace_text:
        return ""
    text = re.sub(r"^\s*INVESTIGATION SUMMARY\s*", "", trace_text, flags=re.I)
    label_alt = "|".join(re.escape(label) for label in REASONING_TRACE_LABELS)
    pattern = rf"(?:{label_alt}):\s*(.*?)(?=(?:{label_alt}):|\Z)"
    matches = re.findall(pattern, text, flags=re.S | re.I)

    if len(matches) != len(REASONING_TRACE_LABELS):
        return trace_text  # couldn't cleanly parse all four parts — show as-is rather than guess

    parts = []
    for label, content in zip(REASONING_TRACE_LABELS, matches):
        content = content.strip()
        if label == "Decision basis":
            steps = [s.strip() for s in re.split(r"\s*\d+\.\s*", content) if s.strip()]
            if len(steps) > 1:
                content = "\n".join(f"{i + 1}. {s}" for i, s in enumerate(steps))
            # Collapsed by default (native HTML disclosure widget, no extra Gradio
            # component needed) since this is usually the longest section and the
            # main reason the panel needed page scrolling to read in full.
            parts.append(
                f"<details><summary><strong>{label}</strong> (click to expand)</summary>\n\n"
                f"{content}\n\n</details>"
            )
        else:
            parts.append(f"**{label}:** {content}")
    return "\n\n".join(parts)


def render_case_panel(case_state):
    """Builds the panel display values from case_state. Once a case closes, the
    second panel switches from Hypotheses to Recommendation. The third panel switches
    between Missing (while the case is open) and Reasoning Trace (once closed, if the
    case was closed with Reasoning Mode on) — one always-visible column whose content
    changes, rather than two separate hide/show components. That avoids a layout bug
    where a hidden component's column still reserved its share of the row's width,
    leaving a visible gap wherever the hidden one happened to be."""
    status_html = render_status_html(case_state)
    is_closed = case_state["status"] == "CLOSED"

    evidence_list = case_state["evidence"]
    evidence_md = "### Evidence\n" + ("\n".join(f"- {e}" for e in evidence_list) if evidence_list else "_No evidence yet._")

    if is_closed:
        second_panel_md = "### Recommendation\n" + (case_state["recommendation"] or "_No recommendation available._")
    else:
        second_panel_md = "### Hypotheses\n" + (case_state["hypotheses"] or "_No hypotheses yet._")

    has_trace = bool(case_state.get("reasoning_trace"))
    if has_trace:
        third_panel_md = "### Reasoning Trace\n\n" + format_reasoning_trace(case_state["reasoning_trace"])
        third_classes = ["reasoning-panel"]
    elif is_closed:
        third_panel_md = ""  # Missing is no longer relevant once closed, with or without a trace
        third_classes = []
    else:
        third_panel_md = "### Missing\n" + (case_state["missing"] or "_Nothing flagged yet._")
        third_classes = []
    third_update = gr.update(value=third_panel_md, elem_classes=third_classes)

    return status_html, evidence_md, second_panel_md, third_update


CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Special+Elite&family=Inter:wght@400;500;600&display=swap');

.gradio-container {
    background: radial-gradient(circle at top, #2d1b4e 0%, #1a1025 45%, #0d0812 100%) !important;
    font-family: 'Inter', sans-serif !important;
    position: relative;
}

.gradio-container::after {
    content: "";
    position: fixed;
    inset: 0;
    pointer-events: none;
    box-shadow: inset 0 0 220px rgba(0, 0, 0, 0.55);
}

.main-row {
    align-items: stretch !important;
}

.sections-wrapper {
    max-width: 1100px;
    margin: 0 auto !important;
    width: 100%;
}

.sections-row {
    gap: 24px;
    flex-wrap: wrap;
}

.sections-row > * {
    flex-basis: 200px;
}

.spotlight {
    background: radial-gradient(ellipse at center, rgba(168, 85, 247, 0.35) 0%, rgba(168, 85, 247, 0) 70%);
    padding: 16px 0 0 0;
}

#interrogator-header {
    font-family: 'Special Elite', monospace;
    font-size: 2em;
    letter-spacing: 3px;
    text-transform: uppercase;
    color: #e9d5ff;
    text-shadow: 0 0 18px rgba(168, 85, 247, 0.7);
    text-align: center;
    margin-bottom: 4px;
}

#interrogator-subheader {
    font-size: 1.15em;
    text-align: center;
    color: #ddd6fe;
    opacity: 0.9;
    margin-bottom: 0;
}

.case-file-card {
    background: rgba(46, 16, 89, 0.55) !important;
    border: 1px solid rgba(196, 158, 255, 0.6) !important;
    border-radius: 14px !important;
    padding: 18px !important;
    box-shadow: 0 0 28px rgba(139, 92, 246, 0.35), inset 0 0 40px rgba(0, 0, 0, 0.25);
    position: relative;
    box-sizing: border-box;
    height: 100%;
    align-self: stretch;
}

.case-file-stamp {
    position: absolute;
    top: 10px;
    right: 20px;
    border: 2px solid #f87171;
    color: #f87171;
    font-family: 'Special Elite', monospace;
    font-size: 0.7em;
    letter-spacing: 2px;
    padding: 3px 8px;
    border-radius: 4px;
    transform: rotate(8deg);
    opacity: 0.85;
}

.case-file-card h3 {
    font-family: 'Special Elite', monospace;
    letter-spacing: 1.5px;
    color: #f5f0ff !important;
    border-bottom: 1px dashed rgba(216, 180, 254, 0.5);
    padding-bottom: 6px;
    margin-top: 8px;
    margin-bottom: 8px;
}

.case-file-card p,
.case-file-card li,
.case-file-card em,
.case-file-card span {
    color: #ede9fe !important;
    font-family: 'Special Elite', monospace;
    font-size: 0.92em;
    line-height: 1.5;
}

.status-row {
    display: flex;
    flex-direction: row;
    align-items: center;
    gap: 18px;
    margin-bottom: 14px;
    flex-wrap: wrap;
}

.status-badge {
    border-radius: 999px;
    padding: 10px 18px;
    text-align: center;
    font-weight: 700;
    letter-spacing: 1px;
    color: white;
    box-shadow: 0 2px 10px rgba(0, 0, 0, 0.4);
    flex: 1;
}

.interrogator-figure {
    display: flex;
    flex-direction: column;
    align-items: center;
    flex-shrink: 0;
    margin: 0;
}

.interrogator-figure svg {
    width: 100%;
    max-width: 120px;
    height: auto;
    overflow: visible;
}

.figure-caption {
    font-family: 'Special Elite', monospace;
    color: #ddd6fe;
    letter-spacing: 2px;
    text-transform: uppercase;
    font-size: 0.72em;
    margin-top: 2px;
}

.lamp-glow { animation: lamp-pulse 3s ease-in-out infinite; }
.det-head { animation: head-bob 1.8s ease-in-out infinite alternate; }
.magnifier-arm { animation: magnifier-sweep 2.6s ease-in-out infinite; }
.qmark { font-family: 'Special Elite', monospace; font-size: 16px; opacity: 0; animation: qmark-float 3.2s ease-in infinite; }
.q2 { animation-delay: 1s; }
.q3 { animation-delay: 2s; }

.hat-tip { animation: hat-tip 0.9s ease-out 0.9s both; }
.stamp-slam { animation: stamp-slam 0.55s cubic-bezier(0.2, 0.9, 0.3, 1.25) 0.25s both; }
.folder-thump { animation: folder-thump 0.6s ease-out 0.5s both; }
.impact { opacity: 0; animation: impact-flash 0.4s ease-out 0.55s both; }

@keyframes lamp-pulse { 0%, 100% { opacity: 0.65; } 50% { opacity: 1; } }
@keyframes head-bob { from { transform: translateY(0); } to { transform: translateY(-2.5px); } }
@keyframes magnifier-sweep { 0%, 100% { transform: rotate(-9deg); } 50% { transform: rotate(11deg); } }
@keyframes qmark-float {
    0% { opacity: 0; transform: translateY(8px); }
    25% { opacity: 0.95; }
    100% { opacity: 0; transform: translateY(-16px); }
}
@keyframes hat-tip {
    0% { transform: rotate(0deg); }
    40% { transform: rotate(-12deg) translateY(-3px); }
    100% { transform: rotate(-6deg) translateY(-1px); }
}
@keyframes stamp-slam {
    0% { transform: scale(2.8) rotate(-12deg); opacity: 0; }
    55% { transform: scale(0.9) rotate(-12deg); opacity: 1; }
    75% { transform: scale(1.07) rotate(-12deg); }
    100% { transform: scale(1) rotate(-12deg); opacity: 1; }
}
@keyframes folder-thump { 0% { transform: translateY(0); } 40% { transform: translateY(3px); } 100% { transform: translateY(0); } }
@keyframes impact-flash { 0% { opacity: 0; } 30% { opacity: 1; } 100% { opacity: 0; } }

@media (prefers-reduced-motion: reduce) {
    .lamp-glow, .det-head, .magnifier-arm, .qmark, .folder-thump, .impact, .hat-tip { animation: none !important; }
    .stamp-slam { animation: none !important; transform: rotate(-12deg); }
    .qmark { opacity: 0.7; }
}

.reasoning-panel {
    background: rgba(15, 76, 92, 0.35) !important;
    border: 1px solid rgba(94, 234, 212, 0.55) !important;
    border-radius: 12px !important;
    padding: 14px !important;
    box-shadow: 0 0 18px rgba(45, 212, 191, 0.25);
    height: 100%;
    box-sizing: border-box;
}

.reasoning-panel h3 {
    color: #99f6e4 !important;
    border-bottom-color: rgba(94, 234, 212, 0.4) !important;
}

.reasoning-panel p, .reasoning-panel li {
    color: #ccfbf1 !important;
}

.reasoning-panel strong, .reasoning-panel b {
    color: #5eead4 !important;
}

.reasoning-panel summary {
    cursor: pointer;
    color: #5eead4;
    margin-bottom: 6px;
}

.reasoning-panel details {
    margin-top: 4px;
}

.reasoning-toggle {
    background: rgba(15, 76, 92, 0.4) !important;
    border: 1px solid rgba(94, 234, 212, 0.6) !important;
    border-radius: 10px !important;
    box-shadow: 0 0 14px rgba(45, 212, 191, 0.3);
    padding: 6px 10px !important;
}

.reasoning-toggle label {
    color: #99f6e4 !important;
    font-weight: 600;
}

.reasoning-toggle input[type="checkbox"] {
    accent-color: #2dd4bf;
    width: 18px;
    height: 18px;
}

/* Investigation summary rendered as a blockquote in the chat itself, visually distinct from the final conclusion */
.message blockquote {
    border-left: 3px solid #2dd4bf;
    background: rgba(15, 76, 92, 0.15);
    padding: 6px 12px;
    margin: 8px 0;
    border-radius: 4px;
}
"""

THEME = gr.themes.Soft(primary_hue="violet", secondary_hue="purple", neutral_hue="slate")

with gr.Blocks(title="The Interrogator") as demo:
    with gr.Row(elem_classes=["main-row"]):
        with gr.Column(scale=1):
            chatbot = gr.Chatbot()
            msg = gr.Textbox(label="Your message", placeholder="Present your case...")
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                new_case_btn = gr.Button("New Case")
                reasoning_toggle = gr.Checkbox(
                    label="Reasoning Mode", value=True, elem_classes=["reasoning-toggle"]
                )
        with gr.Column(scale=2, elem_classes=["case-file-card"]):
            gr.HTML('<div class="case-file-stamp">CONFIDENTIAL</div>')
            gr.HTML(
                '<div class="spotlight">'
                '<div id="interrogator-header">The Interrogator</div>'
                '<div id="interrogator-subheader">Present your case. The case file updates live as the detective works.</div>'
                "</div>"
            )
            gr.Markdown("### Case File")
            status_display = gr.HTML()
            with gr.Column(elem_classes=["sections-wrapper"]):
                with gr.Row(elem_classes=["sections-row"]):
                    with gr.Column(scale=1, min_width=140):
                        evidence_panel = gr.Markdown("### Evidence\n_No evidence yet._")
                    with gr.Column(scale=1, min_width=140):
                        second_panel = gr.Markdown("### Hypotheses\n_No hypotheses yet._")
                    with gr.Column(scale=1, min_width=140):
                        third_panel = gr.Markdown("### Missing\n_Nothing flagged yet._")

    case_state_box = gr.State(new_case_state())

    def handle_send(message, chat_history, case_state, reasoning_mode):
        if not message or not message.strip():
            status_html, ev, hyp, thr = render_case_panel(case_state)
            return chat_history, case_state, "", status_html, ev, hyp, thr
        reply, case_state = stateful_investigate(message, case_state, reasoning_mode)
        chat_history = chat_history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply},
        ]
        status_html, ev, hyp, thr = render_case_panel(case_state)
        return chat_history, case_state, "", status_html, ev, hyp, thr

    def handle_new_case(chat_history, case_state):
        fresh_chat = [{"role": "assistant", "content": "New case started. What's going on?"}]
        case_state = new_case_state()
        status_html, ev, hyp, thr = render_case_panel(case_state)
        return fresh_chat, case_state, status_html, ev, hyp, thr

    msg.submit(
        handle_send,
        [msg, chatbot, case_state_box, reasoning_toggle],
        [chatbot, case_state_box, msg, status_display, evidence_panel, second_panel, third_panel],
    )
    send_btn.click(
        handle_send,
        [msg, chatbot, case_state_box, reasoning_toggle],
        [chatbot, case_state_box, msg, status_display, evidence_panel, second_panel, third_panel],
    )
    new_case_btn.click(
        handle_new_case,
        [chatbot, case_state_box],
        [chatbot, case_state_box, status_display, evidence_panel, second_panel, third_panel],
    )

    demo.load(lambda cs: render_status_html(cs), [case_state_box], [status_display])

demo.launch(share=True, debug=True, theme=THEME, css=CUSTOM_CSS, inline=False)